# Bygning af billedgenereringsapplikationer (OpenAI)

Der er mere ved LLM'er end tekstgenerering. Du kan også generere billeder fra tekstbeskrivelser. Billeder som en modalitet er nyttige inden for MedTech, arkitektur, turisme, spiludvikling, markedsføring og mere. I denne lektion arbejder vi med nutidens **GPT Image** modeller på OpenAI platformen.

## Læringsmål

Når du er færdig med denne lektion, vil du kunne:

- Forklare hvad billedgenerering er, og hvor det er nyttigt.
- Forstå `gpt-image` modelfamilien og hvordan den adskiller sig fra de ældre DALL·E-modeller.
- Bygge en billedgenereringsapplikation og redigere billeder.

## Hvad er billedgenerering?

Billedgenereringsmodeller skaber billeder ud fra en tekstprompt. Moderne modeller som `gpt-image` lærer forholdet mellem tekst og billeder under træningen, og omsætter derefter tilfældig støj til et billede, der matcher din prompt.

To velkendte familier af billedmodeller er:

- **`gpt-image` (OpenAI)** — den nuværende generation, der bruges i denne lektion. Den understøtter tekst-til-billede generering og billedredigering (inpainting med en maske).
- **Midjourney** — en populær tredjepartsmodel med sin egen service og Discord-baserede arbejdsgang.

> De ældre OpenAI billedmodeller — **DALL·E 2** og **DALL·E 3** — er forældede. DALL·E 3 er ikke længere tilgængelig for nye implementeringer, og funktioner som `create_variation` fandtes kun i DALL·E 2. Brug `gpt-image` modellerne til nye applikationer.

> **Vigtigt:** `gpt-image` modeller returnerer det genererede billede som **base64** (`b64_json`), ikke som en URL. Din kode dekoder base64-strengen til bytes og gemmer det — der er ingen billed-URL til download.


## Byg din første billedgenereringsapplikation

Så hvad kræver det at bygge en billedgenereringsapplikation? Du har brug for følgende biblioteker:

- **python-dotenv**, det anbefales kraftigt at bruge dette bibliotek til at holde dine hemmeligheder i en *.env*-fil adskilt fra koden.
- **openai**, dette bibliotek bruger du til at interagere med OpenAI API'en.
- **pillow**, til at arbejde med billeder i Python.


1. Opret en fil *.env* med følgende indhold:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Saml ovenstående biblioteker i en fil kaldet *requirements.txt* som følger:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Opret derefter et virtuelt miljø og installer bibliotekerne:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> For Windows, brug følgende kommandoer for at oprette og aktivere dit virtuelle miljø:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Tilføj følgende kode i en fil kaldet *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Lad os forklare denne kode:

- Først importerer vi de biblioteker, vi har brug for, inklusive OpenAI biblioteket, dotenv biblioteket, base64 modulet og Pillow biblioteket.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Derefter opretter vi klienten, som læser API-nøglen fra din ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Næste genererer vi billedet:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` modeller returnerer billedet som en **base64** streng i `data[0].b64_json`. Vi dekoder det til bytes og skriver det til en fil — der er ingen URL til download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Til sidst åbner vi billedet og bruger standard billedfremviser til at vise det:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Flere detaljer om at generere billedet

Lad os se på parametrene for `images.generate`:

- **model**, er billedmodellen, der skal bruges (for eksempel `gpt-image-2`).
- **prompt**, er tekstprompten brugt til at generere billedet. Her er det "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, er størrelsen på det genererede billede (for eksempel `1024x1024`, `1536x1024`, `1024x1536`, eller `"auto"`).
- **n**, er antallet af genererede billeder. Her genererer vi ét.

> Billedmodeller tager ikke en `temperature` parameter — det er en kontrol til tekstgenerering. For at få variation, kald API'en igen; for at reducere variation, gør din prompt mere specifik.

## Yderligere funktioner ved billedgenerering

Du har set, hvordan man genererer et billede med få linjer Python. `gpt-image` modeller kan også **redigere** et eksisterende billede. Ved at give et billede, en valgfri **maske** (som markerer området der skal ændres), og en prompt, kan du ændre en del af et billede — for eksempel tilføje en hat til vores kanin.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Basisbilledet indeholder bare kaninen; det endelige billede har hatten på kaninen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
